# Module 3.3: Transformer Decoder 解码器

## 1. 本章概览 (Overview)

### 📚 学习目标

1. **掩码自注意力**：理解因果掩码和自回归生成
2. **交叉注意力**：理解解码器如何关注编码器输出
3. **解码器层**：实现完整的 Transformer 解码器层
4. **完整 Transformer**：构建编码器-解码器架构
5. **生成策略**：实现贪婪解码和束搜索

### 🎯 核心问题

- 为什么解码器需要掩码注意力？
- 交叉注意力如何工作？
- 如何实现自回归生成？
- Transformer 如何用于序列到序列任务？

### 🏢 业务场景

本章技术将应用于 `电商客服智能助理` 场景：
- 客服回复为什么会出现重复或矛盾？→ 自回归生成与注意力掩码
- 如何控制回复的风格和长度？→ 解码器的生成策略调优

### 🗺️ 架构图

```
编码器-解码器 Transformer

输入序列                     输出序列（移位）
    ↓                            ↓
[编码器堆栈]  ←────────→  [解码器堆栈]
    ↓                            ↓
编码器输出                   预测下一个词
```

### ⏱️ 预计学习时间：3-4小时

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import math

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

# Configure matplotlib
plt.rcParams['figure.figsize'] = (12, 6)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")
print(f"✓ PyTorch version: {torch.__version__}")

## 2. 动机与背景 (Motivation)

### 为什么需要 Transformer Decoder？

在序列到序列任务中（如机器翻译），我们需要：

1. **编码器**：理解输入序列（源语言）
2. **解码器**：生成输出序列（目标语言）

### 解码器的特殊要求

**自回归生成 (Autoregressive Generation)** 是一种序列生成方式，模型逐步生成序列中的每个元素，每一步的输出仅依赖于之前已生成的所有元素。在本章场景中，它是 Transformer 解码器生成目标序列的核心机制。

**具体而言**：解码器一次生成一个词，每个词只能看到之前生成的词，不能看到未来的词。

```
生成过程：
Step 1: <BOS> → 我
Step 2: <BOS> 我 → 爱
Step 3: <BOS> 我 爱 → 深度
Step 4: <BOS> 我 爱 深度 → 学习
```

这就需要**掩码注意力**来防止看到未来的词。

## 3. 理论基础 (Theory)

### 3.1 掩码自注意力 (Masked Self-Attention)

**因果掩码 (Causal Mask)** 是一种下三角形式的注意力掩码矩阵，强制每个位置只能关注自身及之前的位置，屏蔽未来位置的信息。在本章场景中，它用于保证 Transformer 解码器的自回归特性——训练时防止模型"偷看"未来的词。

**因果掩码的数学表示**：

$$\text{mask}_{ij} = \begin{cases} 0 & \text{if } i \geq j \\ -\infty & \text{if } i < j \end{cases}$$

- 位置 $i$ 只能看到位置 $j \leq i$ 的信息
- 形成下三角矩阵模式

**掩码注意力计算**：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + \text{Mask}\right)V$$

其中 Mask 在不允许关注的位置设为 $-\infty$，softmax 后变为 0。

In [ ]:
# 🔬 Micro Practice: Visualize Causal Mask
# Goal: Understand the triangular mask pattern
# Expected outcome: See which positions can attend to which

def create_causal_mask(seq_len):
    """
    Create causal mask for autoregressive generation
    
    Args:
        seq_len: Sequence length
    
    Returns:
        mask: Causal mask (seq_len, seq_len)
    """
    # Create lower triangular matrix (1s below diagonal, 0s above)
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# Visualize causal mask
seq_len = 8
mask = create_causal_mask(seq_len)

plt.figure(figsize=(8, 8))
plt.imshow(mask, cmap='Blues', aspect='auto')
plt.colorbar(label='Can Attend (1=Yes, 0=No)')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.title('Causal Mask Pattern\n(Each row shows what that position can attend to)')

# Add grid
for i in range(seq_len + 1):
    plt.axhline(i - 0.5, color='gray', linewidth=0.5)
    plt.axvline(i - 0.5, color='gray', linewidth=0.5)

# Add labels
plt.xticks(range(seq_len), [f't{i}' for i in range(seq_len)])
plt.yticks(range(seq_len), [f't{i}' for i in range(seq_len)])

plt.tight_layout()
plt.show()

print("✓ 因果掩码可视化完成")
print(f"\\n解释：")
print(f"- 位置 t0 只能看到 t0（自己）")
print(f"- 位置 t1 可以看到 t0, t1")
print(f"- 位置 t2 可以看到 t0, t1, t2")
print(f"- 以此类推...")

In [ ]:
# 🔬 Micro Practice: Implement Masked Self-Attention
# Goal: Build self-attention with causal masking
# Expected outcome: Attention respects causal constraint

def masked_self_attention(Q, K, V, mask=None):
    """
    Compute masked self-attention
    
    Args:
        Q: Query matrix (batch, seq_len, d_k)
        K: Key matrix (batch, seq_len, d_k)
        V: Value matrix (batch, seq_len, d_v)
        mask: Causal mask (seq_len, seq_len)
    
    Returns:
        output: Attention output (batch, seq_len, d_v)
        attention_weights: Attention weights (batch, seq_len, seq_len)
    """
    d_k = Q.size(-1)
    
    # Compute attention scores
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply mask (set masked positions to -inf)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    
    # Apply softmax
    attention_weights = F.softmax(scores, dim=-1)
    
    # Handle NaN from softmax(-inf)
    attention_weights = torch.nan_to_num(attention_weights, 0.0)
    
    # Compute output
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

# Test masked self-attention
batch_size = 2
seq_len = 6
d_model = 64

# Create random Q, K, V
Q = torch.randn(batch_size, seq_len, d_model)
K = torch.randn(batch_size, seq_len, d_model)
V = torch.randn(batch_size, seq_len, d_model)

# Create causal mask
mask = create_causal_mask(seq_len).unsqueeze(0)  # (1, seq_len, seq_len)

# Compute masked attention
output, attn_weights = masked_self_attention(Q, K, V, mask)

print("Masked Self-Attention Test:")
print(f"Input shape: {Q.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print()

# Visualize attention pattern
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(mask[0].numpy(), cmap='Blues', aspect='auto')
plt.title('Causal Mask')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(attn_weights[0].detach().numpy(), cmap='viridis', aspect='auto')
plt.title('Attention Weights (with mask)')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.colorbar()

plt.tight_layout()
plt.show()

print("✓ 掩码自注意力实现成功！")
print("注意：注意力权重只在掩码允许的位置有值")

### 3.2 编码器-解码器注意力 (Cross-Attention)

**交叉注意力 (Cross-Attention)** 是一种注意力机制，其 Query 来自一个序列，而 Key 和 Value 来自另一个序列，用于建立两个不同序列之间的关联。在本章场景中，它让 Transformer 解码器能够在生成每个词时动态关注编码器输出（源序列）的相关部分，实现源语言到目标语言的对齐。

**交叉注意力机制**：

在解码器中，除了自注意力，还需要关注编码器的输出：

- **Query (Q)**：来自解码器（"我想要什么信息？"）
- **Key (K)** 和 **Value (V)**：来自编码器（"源序列有什么信息？"）

$$\\text{CrossAttention}(Q_{dec}, K_{enc}, V_{enc}) = \\text{softmax}\\left(\\frac{Q_{dec}K_{enc}^T}{\\sqrt{d_k}}\\right)V_{enc}$$

**作用**：
- 解码器在生成每个词时，可以关注源序列的不同部分
- 类似于 Seq2Seq 中的注意力机制
- 实现源语言和目标语言之间的对齐

**与自注意力的区别**：
- 自注意力：Q, K, V 都来自同一个序列
- 交叉注意力：Q 来自解码器，K 和 V 来自编码器

In [ ]:
# 🔬 Micro Practice: Implement Cross-Attention
# Goal: Build encoder-decoder attention
# Expected outcome: Decoder attends to encoder outputs

def cross_attention(Q_dec, K_enc, V_enc):
    """
    Compute cross-attention (encoder-decoder attention)
    
    Args:
        Q_dec: Query from decoder (batch, tgt_len, d_k)
        K_enc: Key from encoder (batch, src_len, d_k)
        V_enc: Value from encoder (batch, src_len, d_v)
    
    Returns:
        output: Attention output (batch, tgt_len, d_v)
        attention_weights: Attention weights (batch, tgt_len, src_len)
    """
    d_k = Q_dec.size(-1)
    
    # Compute attention scores
    scores = torch.matmul(Q_dec, K_enc.transpose(-2, -1)) / math.sqrt(d_k)
    
    # Apply softmax (no mask needed for cross-attention)
    attention_weights = F.softmax(scores, dim=-1)
    
    # Compute output
    output = torch.matmul(attention_weights, V_enc)
    
    return output, attention_weights

# Test cross-attention
batch_size = 2
src_len = 8  # Source sequence length (encoder)
tgt_len = 6  # Target sequence length (decoder)
d_model = 64

# Simulate encoder outputs (K and V from encoder)
K_enc = torch.randn(batch_size, src_len, d_model)
V_enc = torch.randn(batch_size, src_len, d_model)

# Simulate decoder queries (Q from decoder)
Q_dec = torch.randn(batch_size, tgt_len, d_model)

# Compute cross-attention
output, attn_weights = cross_attention(Q_dec, K_enc, V_enc)

print("Cross-Attention Test:")
print(f"Decoder query shape: {Q_dec.shape}")
print(f"Encoder key shape: {K_enc.shape}")
print(f"Encoder value shape: {V_enc.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")
print()

# Visualize cross-attention pattern
plt.figure(figsize=(10, 6))
plt.imshow(attn_weights[0].detach().numpy(), cmap='viridis', aspect='auto')
plt.colorbar(label='Attention Weight')
plt.xlabel('Source Position (Encoder)')
plt.ylabel('Target Position (Decoder)')
plt.title('Cross-Attention Pattern\\n(Decoder attending to Encoder)')

# Add grid
for i in range(tgt_len + 1):
    plt.axhline(i - 0.5, color='white', linewidth=0.5, alpha=0.3)
for j in range(src_len + 1):
    plt.axvline(j - 0.5, color='white', linewidth=0.5, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ 交叉注意力实现成功！")
print("\\n解释：")
print("- 每一行显示解码器的一个位置关注编码器的哪些位置")
print("- 不同于自注意力，这里没有因果掩码")
print("- 解码器可以看到编码器的所有位置")

### 3.3 解码器层 (Decoder Layer)

**完整解码器层结构**：

```
输入
  ↓
[1. 掩码多头自注意力] ← 看之前的输出
  ↓
[Add & Norm]
  ↓
[2. 多头交叉注意力] ← 看编码器输出
  ↓
[Add & Norm]
  ↓
[3. 前馈网络 (FFN)]
  ↓
[Add & Norm]
  ↓
输出
```

**三个子层**：

1. **掩码自注意力**：解码器关注自己之前生成的内容
2. **交叉注意力**：解码器关注编码器的输出
3. **前馈网络**：对每个位置独立进行非线性变换

每个子层后都有：
- **残差连接**：`output = sublayer(x) + x`
- **层归一化**：`output = LayerNorm(output)`

In [ ]:
# 🔬 Micro Practice: Implement Decoder Layer
# Goal: Build complete decoder layer with all three sub-layers
# Expected outcome: Full decoder layer with masked attention, cross-attention, and FFN

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention module"""
    
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        assert d_model % num_heads == 0
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)
        
        # Linear projections and split into heads
        Q = self.W_q(query).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Compute attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = torch.nan_to_num(attn_weights, 0.0)
        
        output = torch.matmul(attn_weights, V)
        
        # Concatenate heads and apply final linear
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        output = self.W_o(output)
        
        return output, attn_weights

class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network"""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        return self.linear2(self.dropout(F.relu(self.linear1(x))))

class DecoderLayer(nn.Module):
    """Single Transformer Decoder Layer"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(DecoderLayer, self).__init__()
        
        # Three sub-layers
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        Args:
            x: Decoder input (batch, tgt_len, d_model)
            encoder_output: Encoder output (batch, src_len, d_model)
            src_mask: Source mask for cross-attention
            tgt_mask: Target mask for self-attention (causal)
        """
        # 1. Masked self-attention
        attn_output, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # 2. Cross-attention with encoder
        attn_output, _ = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.dropout(attn_output))
        
        # 3. Feed-forward
        ff_output = self.feed_forward(x)
        x = self.norm3(x + self.dropout(ff_output))
        
        return x

# Test decoder layer
d_model = 512
num_heads = 8
d_ff = 2048
batch_size = 2
src_len = 10
tgt_len = 8

decoder_layer = DecoderLayer(d_model, num_heads, d_ff)

# Create dummy inputs
decoder_input = torch.randn(batch_size, tgt_len, d_model)
encoder_output = torch.randn(batch_size, src_len, d_model)
tgt_mask = create_causal_mask(tgt_len)

# Forward pass
output = decoder_layer(decoder_input, encoder_output, tgt_mask=tgt_mask)

print("Decoder Layer Test:")
print(f"Decoder input shape: {decoder_input.shape}")
print(f"Encoder output shape: {encoder_output.shape}")
print(f"Output shape: {output.shape}")
print("✓ 解码器层实现成功！")

## 4. 实现细节 (Implementation)

### 4.1 完整解码器堆栈

**解码器堆栈结构**：

```
输入嵌入 + 位置编码
    ↓
[解码器层 1]
    ↓
[解码器层 2]
    ↓
    ...
    ↓
[解码器层 N]
    ↓
线性投影 + Softmax
    ↓
输出概率分布
```

**关键组件**：
1. **输入嵌入**：将词索引转换为向量
2. **位置编码**：添加位置信息
3. **N 个解码器层**：堆叠多个解码器层
4. **输出投影**：将隐藏状态映射到词汇表大小

**自回归生成过程**：
- 从 `<BOS>` 开始
- 每次生成一个词
- 将生成的词作为下一步的输入
- 直到生成 `<EOS>` 或达到最大长度

In [ ]:
# 🔬 Micro Practice: Implement Full Decoder
# Goal: Build complete decoder with N stacked layers
# Expected outcome: Full decoder stack with embeddings and output projection

class PositionalEncoding(nn.Module):
    """Positional Encoding using sine and cosine functions"""
    
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """Add positional encoding to input"""
        return x + self.pe[:, :x.size(1), :]

class TransformerDecoder(nn.Module):
    """Complete Transformer Decoder"""
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1, max_len=5000):
        super(TransformerDecoder, self).__init__()
        
        self.d_model = d_model
        
        # Embedding and positional encoding
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        # Stack of decoder layers
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Output projection
        self.output_projection = nn.Linear(d_model, vocab_size)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        """
        Args:
            tgt: Target sequence (batch, tgt_len)
            encoder_output: Encoder output (batch, src_len, d_model)
            src_mask: Source mask
            tgt_mask: Target causal mask
        
        Returns:
            output: Logits (batch, tgt_len, vocab_size)
        """
        # Embedding and positional encoding
        x = self.embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        
        # Pass through decoder layers
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        
        # Output projection
        output = self.output_projection(x)
        
        return output

# Test full decoder
vocab_size = 10000
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6
batch_size = 2
src_len = 10
tgt_len = 8

decoder = TransformerDecoder(vocab_size, d_model, num_heads, d_ff, num_layers)

# Create dummy inputs
tgt = torch.randint(0, vocab_size, (batch_size, tgt_len))
encoder_output = torch.randn(batch_size, src_len, d_model)
tgt_mask = create_causal_mask(tgt_len)

# Forward pass
output = decoder(tgt, encoder_output, tgt_mask=tgt_mask)

print("Full Decoder Test:")
print(f"Target input shape: {tgt.shape}")
print(f"Encoder output shape: {encoder_output.shape}")
print(f"Decoder output shape: {output.shape}")
print(f"Number of decoder layers: {num_layers}")
print("✓ 完整解码器堆栈实现成功！")

In [ ]:
# 🔬 Micro Practice: Demonstrate Autoregressive Generation
# Goal: Show step-by-step generation process
# Expected outcome: Understand how decoder generates one token at a time

def greedy_decode(decoder, encoder_output, start_token, max_len=20):
    """
    Greedy decoding: generate one token at a time
    
    Args:
        decoder: Trained decoder
        encoder_output: Encoder output (1, src_len, d_model)
        start_token: Start token ID (e.g., <BOS>)
        max_len: Maximum generation length
    
    Returns:
        generated: Generated sequence
    """
    decoder.eval()
    
    # Start with start token
    generated = torch.LongTensor([[start_token]])
    
    with torch.no_grad():
        for _ in range(max_len):
            # Create causal mask
            tgt_len = generated.size(1)
            tgt_mask = create_causal_mask(tgt_len)
            
            # Forward pass
            output = decoder(generated, encoder_output, tgt_mask=tgt_mask)
            
            # Get next token (greedy: argmax)
            next_token = output[:, -1, :].argmax(dim=-1)
            
            # Append to generated sequence
            generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)
            
            # Stop if we generate end token (assume 2 is <EOS>)
            if next_token.item() == 2:
                break
    
    return generated

# Demonstrate generation process
print("自回归生成演示：")
print("=" * 60)

# Simulate encoder output
encoder_output = torch.randn(1, 10, d_model)

# Generate sequence
start_token = 1  # <BOS>
generated = greedy_decode(decoder, encoder_output, start_token, max_len=10)

print(f"生成过程：")
print(f"Step 0: [<BOS>] (token {start_token})")
for i in range(1, generated.size(1)):
    print(f"Step {i}: {generated[0, :i+1].tolist()}")

print()
print("✓ 自回归生成演示完成！")
print("\\n说明：")
print("- 每一步只能看到之前生成的词（因果掩码）")
print("- 使用贪婪解码：选择概率最高的词")
print("- 实际应用中可以使用束搜索（Beam Search）获得更好的结果")

### 4.2 完整 Transformer 模型

**编码器-解码器架构**：

```
源序列                     目标序列
  ↓                          ↓
[输入嵌入]              [输出嵌入]
  ↓                          ↓
[位置编码]              [位置编码]
  ↓                          ↓
[编码器层 × N]          [解码器层 × N]
  ↓                          ↓  ↑
  └──────────────────────────┘  │
         (交叉注意力)            │
                                │
                          [线性 + Softmax]
                                ↓
                            输出概率
```

**完整流程**：

1. **编码阶段**：
   - 源序列 → 嵌入 + 位置编码
   - 通过 N 个编码器层
   - 得到编码器输出（记忆）

2. **解码阶段**：
   - 目标序列 → 嵌入 + 位置编码
   - 通过 N 个解码器层
   - 每层都关注编码器输出
   - 输出投影到词汇表

3. **训练**：Teacher Forcing
4. **推理**：自回归生成

In [ ]:
# 🔬 Micro Practice: Implement Complete Transformer
# Goal: Build full encoder-decoder Transformer
# Expected outcome: Complete seq2seq model

class EncoderLayer(nn.Module):
    """Single Transformer Encoder Layer"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(EncoderLayer, self).__init__()
        
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Self-attention
        attn_output, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x

class TransformerEncoder(nn.Module):
    """Complete Transformer Encoder"""
    
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers, dropout=0.1, max_len=5000):
        super(TransformerEncoder, self).__init__()
        
        self.d_model = d_model
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, src, src_mask=None):
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        
        for layer in self.layers:
            x = layer(x, src_mask)
        
        return x

class Transformer(nn.Module):
    """Complete Transformer Model (Encoder-Decoder)"""
    
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8, 
                 d_ff=2048, num_layers=6, dropout=0.1, max_len=5000):
        super(Transformer, self).__init__()
        
        self.encoder = TransformerEncoder(src_vocab_size, d_model, num_heads, d_ff, 
                                         num_layers, dropout, max_len)
        self.decoder = TransformerDecoder(tgt_vocab_size, d_model, num_heads, d_ff, 
                                         num_layers, dropout, max_len)
    
    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        """
        Args:
            src: Source sequence (batch, src_len)
            tgt: Target sequence (batch, tgt_len)
            src_mask: Source mask
            tgt_mask: Target causal mask
        
        Returns:
            output: Logits (batch, tgt_len, tgt_vocab_size)
        """
        # Encode
        encoder_output = self.encoder(src, src_mask)
        
        # Decode
        output = self.decoder(tgt, encoder_output, src_mask, tgt_mask)
        
        return output

# Test complete Transformer
src_vocab_size = 10000
tgt_vocab_size = 10000
d_model = 512
num_heads = 8
d_ff = 2048
num_layers = 6

transformer = Transformer(src_vocab_size, tgt_vocab_size, d_model, num_heads, 
                         d_ff, num_layers)

# Create dummy inputs
batch_size = 2
src_len = 10
tgt_len = 8

src = torch.randint(0, src_vocab_size, (batch_size, src_len))
tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_len))
tgt_mask = create_causal_mask(tgt_len)

# Forward pass
output = transformer(src, tgt, tgt_mask=tgt_mask)

print("Complete Transformer Test:")
print(f"Source shape: {src.shape}")
print(f"Target shape: {tgt.shape}")
print(f"Output shape: {output.shape}")
print(f"Model parameters: {sum(p.numel() for p in transformer.parameters()):,}")
print("✓ 完整 Transformer 模型实现成功！")

## 5. 工程实践 (Engineering)

### 5.1 训练技巧

**关键技术**：

1. **标签平滑（Label Smoothing）**：
   - 防止模型过度自信
   - 将硬标签 `[0, 0, 1, 0]` 软化为 `[ε, ε, 1-3ε, ε]`

2. **学习率预热（Warmup）**：
   - 开始时使用小学习率
   - 逐渐增加到目标学习率
   - 然后按计划衰减

3. **梯度裁剪（Gradient Clipping）**：
   - 防止梯度爆炸
   - 限制梯度的最大范数

4. **Teacher Forcing**：
   - 训练时使用真实目标序列
   - 加速收敛

### 5.2 与 PyTorch 内置 Transformer 对比

PyTorch 提供了 `nn.Transformer`，我们的实现与之兼容：

```python
# PyTorch 内置
model = nn.Transformer(
    d_model=512,
    nhead=8,
    num_encoder_layers=6,
    num_decoder_layers=6,
    dim_feedforward=2048
)
```

In [ ]:
# 🔬 Micro Practice: Training Utilities
# Goal: Implement key training components
# Expected outcome: Label smoothing, learning rate scheduler

class LabelSmoothing(nn.Module):
    """Label Smoothing for better generalization"""
    
    def __init__(self, vocab_size, padding_idx, smoothing=0.1):
        super(LabelSmoothing, self).__init__()
        self.criterion = nn.KLDivLoss(reduction='sum')
        self.padding_idx = padding_idx
        self.confidence = 1.0 - smoothing
        self.smoothing = smoothing
        self.vocab_size = vocab_size
    
    def forward(self, x, target):
        """
        Args:
            x: Predictions (batch * seq_len, vocab_size) - log probabilities
            target: Ground truth (batch * seq_len)
        """
        true_dist = torch.zeros_like(x)
        true_dist.fill_(self.smoothing / (self.vocab_size - 2))
        true_dist.scatter_(1, target.unsqueeze(1), self.confidence)
        true_dist[:, self.padding_idx] = 0
        
        mask = (target == self.padding_idx).unsqueeze(1)
        true_dist.masked_fill_(mask, 0.0)
        
        return self.criterion(x, true_dist)

class NoamScheduler:
    """Learning rate scheduler from 'Attention is All You Need'"""
    
    def __init__(self, optimizer, d_model, warmup_steps=4000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0
    
    def step(self):
        """Update learning rate"""
        self.step_num += 1
        lr = self.d_model ** (-0.5) * min(
            self.step_num ** (-0.5),
            self.step_num * self.warmup_steps ** (-1.5)
        )
        
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr
        
        return lr

# Visualize learning rate schedule
d_model = 512
warmup_steps = 4000

steps = list(range(1, 20000))
lrs = [d_model ** (-0.5) * min(s ** (-0.5), s * warmup_steps ** (-1.5)) for s in steps]

plt.figure(figsize=(10, 5))
plt.plot(steps, lrs, linewidth=2)
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Noam Learning Rate Schedule\\n(Warmup + Decay)')
plt.axvline(x=warmup_steps, color='r', linestyle='--', label=f'Warmup Steps ({warmup_steps})')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("✓ 训练工具实现完成！")
print("\\n说明：")
print("- 学习率先线性增加（warmup）")
print("- 然后按步数的平方根衰减")
print("- 这有助于训练稳定性和收敛")

### 5.3 完整 Transformer Decoder 训练循环

下面我们用一个简单的序列反转任务来展示完整的 Transformer 训练流程：数据准备、模型初始化、训练循环（带损失跟踪）和评估。

In [ ]:
# 🔬 Micro Practice: 完整 Transformer Decoder 训练循环
# Goal: 在序列反转任务上训练 Transformer，展示完整训练流程
# Expected outcome: 掌握训练循环的各个环节：数据、损失、优化、评估

import random

# ============================================================
# 1. 数据准备：序列反转任务
# ============================================================

PAD, SOS, EOS = 0, 1, 2
VOCAB_SIZE = 13  # 0=PAD, 1=SOS, 2=EOS, 3-12=数字0-9

def generate_reverse_batch(batch_size=32, min_len=3, max_len=8):
    """生成序列反转任务的 mini-batch"""
    src_batch, tgt_batch = [], []
    for _ in range(batch_size):
        seq_len = random.randint(min_len, max_len)
        src = [random.randint(3, 12) for _ in range(seq_len)]
        tgt = [SOS] + src[::-1] + [EOS]
        src_batch.append(src)
        tgt_batch.append(tgt)
    
    # 填充到 batch 内最大长度
    src_max = max(len(s) for s in src_batch)
    tgt_max = max(len(t) for t in tgt_batch)
    
    src_padded = [s + [PAD] * (src_max - len(s)) for s in src_batch]
    tgt_padded = [t + [PAD] * (tgt_max - len(t)) for t in tgt_batch]
    
    return torch.LongTensor(src_padded), torch.LongTensor(tgt_padded)

# ============================================================
# 2. 创建小型 Transformer 模型
# ============================================================

small_d_model = 64
small_transformer = Transformer(
    src_vocab_size=VOCAB_SIZE,
    tgt_vocab_size=VOCAB_SIZE,
    d_model=small_d_model,
    num_heads=4,
    d_ff=256,
    num_layers=2,
    dropout=0.1
)

total_params = sum(p.numel() for p in small_transformer.parameters())
print(f"模型参数量: {total_params:,}")

# ============================================================
# 3. 训练循环
# ============================================================

optimizer = torch.optim.Adam(small_transformer.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

num_epochs = 30
losses = []

print(f"\n开始训练（序列反转任务）...")
print("=" * 50)

small_transformer.train()
for epoch in range(num_epochs):
    # 每轮生成新的训练数据
    src, tgt = generate_reverse_batch(batch_size=64)
    
    # 解码器输入：目标序列去掉最后一个 token（teacher forcing）
    tgt_input = tgt[:, :-1]
    tgt_output = tgt[:, 1:]
    
    # 创建因果掩码
    tgt_mask = create_causal_mask(tgt_input.size(1))
    
    # 前向传播
    logits = small_transformer(src, tgt_input, tgt_mask=tgt_mask)
    
    # 计算损失
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_output.reshape(-1))
    
    # 反向传播
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(small_transformer.parameters(), max_norm=1.0)
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:2d}/{num_epochs}  Loss: {loss.item():.4f}")

print("=" * 50)

# 绘制损失曲线
plt.figure(figsize=(10, 4))
plt.plot(losses, 'b-', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Transformer Decoder Training Loss', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ============================================================
# 4. 评估：测试序列反转
# ============================================================

small_transformer.eval()
print("\n评估结果：")
print("=" * 50)

correct = 0
num_test = 10

for i in range(num_test):
    # 生成测试样本
    seq_len = random.randint(3, 6)
    src_seq = [random.randint(3, 12) for _ in range(seq_len)]
    expected = src_seq[::-1]
    
    src_tensor = torch.LongTensor([src_seq])
    
    # 自回归生成
    generated = [SOS]
    with torch.no_grad():
        encoder_output = small_transformer.encoder(src_tensor)
        
        for _ in range(seq_len + 2):
            tgt_tensor = torch.LongTensor([generated])
            tgt_mask = create_causal_mask(len(generated))
            output = small_transformer.decoder(tgt_tensor, encoder_output, tgt_mask=tgt_mask)
            next_token = output[0, -1, :].argmax().item()
            generated.append(next_token)
            if next_token == EOS:
                break
    
    # 去掉 SOS/EOS
    pred = [t for t in generated if t not in [SOS, EOS, PAD]]
    
    # 转换为可读数字
    src_readable = [t - 3 for t in src_seq]
    exp_readable = [t - 3 for t in expected]
    pred_readable = [t - 3 for t in pred]
    
    match = pred == expected
    correct += match
    mark = "✓" if match else "✗"
    print(f"  {mark} 输入: {src_readable}  期望: {exp_readable}  预测: {pred_readable}")

print(f"\n准确率: {correct}/{num_test} ({correct/num_test*100:.0f}%)")
print("✓ Transformer Decoder 完整训练循环实现完成！")

## 6. 综合项目 (Capstone)

### 6.1 生成策略

**三种常见的解码策略**：

1. **贪婪解码（Greedy Decoding）**：
   - 每步选择概率最高的词
   - 快速但可能不是全局最优

2. **束搜索（Beam Search）**：
   - 保留 k 个最佳候选序列
   - 平衡质量和速度
   - k=5 或 k=10 是常见选择

3. **核采样（Nucleus Sampling）**：
   - 从累积概率达到 p 的最小词集中采样
   - 增加多样性
   - 适合创意生成任务

**对比**：
- 贪婪：最快，但可能重复
- 束搜索：质量好，适合翻译
- 采样：多样性高，适合对话/创作

In [ ]:
# 🔬 Micro Practice: Implement Beam Search
# Goal: Implement beam search decoding
# Expected outcome: Better generation quality than greedy

def beam_search(model, src, start_token, end_token, max_len=50, beam_size=5):
    """
    Beam search decoding
    
    Args:
        model: Transformer model
        src: Source sequence (1, src_len)
        start_token: Start token ID
        end_token: End token ID
        max_len: Maximum generation length
        beam_size: Beam size (number of candidates to keep)
    
    Returns:
        best_sequence: Best generated sequence
    """
    model.eval()
    
    with torch.no_grad():
        # Encode source
        encoder_output = model.encoder(src)
        
        # Initialize beam with start token
        # Each beam item: (sequence, score)
        beams = [([start_token], 0.0)]
        
        for _ in range(max_len):
            candidates = []
            
            for seq, score in beams:
                # Skip if sequence already ended
                if seq[-1] == end_token:
                    candidates.append((seq, score))
                    continue
                
                # Prepare input
                tgt = torch.LongTensor([seq]).to(src.device)
                tgt_mask = create_causal_mask(len(seq)).to(src.device)
                
                # Forward pass
                output = model.decoder(tgt, encoder_output, tgt_mask=tgt_mask)
                
                # Get probabilities for next token
                probs = F.softmax(output[0, -1, :], dim=-1)
                
                # Get top-k tokens
                top_probs, top_indices = torch.topk(probs, beam_size)
                
                # Create new candidates
                for prob, idx in zip(top_probs, top_indices):
                    new_seq = seq + [idx.item()]
                    new_score = score - torch.log(prob).item()  # Negative log likelihood
                    candidates.append((new_seq, new_score))
            
            # Keep top beam_size candidates
            beams = sorted(candidates, key=lambda x: x[1])[:beam_size]
            
            # Check if all beams ended
            if all(seq[-1] == end_token for seq, _ in beams):
                break
        
        # Return best sequence
        best_sequence = beams[0][0]
        return best_sequence

# Demonstrate beam search
print("束搜索演示：")
print("=" * 60)

# Create small model for demo
small_transformer = Transformer(
    src_vocab_size=1000,
    tgt_vocab_size=1000,
    d_model=128,
    num_heads=4,
    d_ff=512,
    num_layers=2
)

# Dummy source
src = torch.randint(0, 1000, (1, 10))
start_token = 1
end_token = 2

# Greedy decoding
greedy_result = greedy_decode(small_transformer.decoder, 
                              small_transformer.encoder(src), 
                              start_token, max_len=10)

# Beam search
beam_result = beam_search(small_transformer, src, start_token, end_token, 
                         max_len=10, beam_size=3)

print(f"贪婪解码结果: {greedy_result[0].tolist()}")
print(f"束搜索结果:   {beam_result}")
print()
print("✓ 束搜索实现完成！")
print("\\n说明：")
print("- 束搜索保留多个候选序列")
print("- 选择总体得分最高的序列")
print("- 通常比贪婪解码质量更好")

### 6.2 Top-K 与 Top-P（核）采样

贪婪解码和束搜索都是**确定性**策略，适合翻译等需要精确输出的任务。而在对话、创意写作等场景中，我们需要**随机性**来产生多样化的文本。

**Top-K 采样**：只保留概率最高的 $k$ 个词，将其余词的概率置零后重新归一化，再从中随机采样。$k$ 越小，输出越保守；$k$ 越大，越多样。

**Top-P（Nucleus）采样**：不固定词数，而是保留累积概率刚好超过阈值 $p$ 的最小词集合。这使得候选集的大小能根据分布的"平坦程度"自适应调整——分布尖锐时只选几个词，分布平坦时选更多词。

| 策略 | 特点 | 适用场景 |
|------|------|----------|
| Greedy | 确定性，最快，但单调 | 分类、短文本 |
| Beam Search | 确定性，质量好 | 翻译、摘要 |
| Top-K | 随机，多样性可控 | 对话、故事生成 |
| Top-P (Nucleus) | 随机，自适应候选集 | 通用生成（GPT 默认） |

In [ ]:
# 🔬 Micro Practice: Top-K 与 Top-P 采样
# Goal: 实现 Top-K 和 Top-P (Nucleus) 采样，对比不同生成策略的输出差异
# Expected outcome: 理解采样策略如何控制生成的多样性

def top_k_sampling(logits, k, temperature=1.0):
    """
    Top-K 采样：只保留概率最高的 k 个 token，其余置为 -inf
    
    Args:
        logits: 模型输出的原始 logits (vocab_size,)
        k: 保留的 token 数量
        temperature: 温度参数，控制分布平滑度
    
    Returns:
        sampled_token: 采样得到的 token index
    """
    # 温度缩放
    scaled_logits = logits / temperature
    
    # 找到 top-k 的阈值，将其余 logits 置为 -inf
    top_k_values, _ = torch.topk(scaled_logits, k)
    threshold = top_k_values[-1]
    scaled_logits[scaled_logits < threshold] = float('-inf')
    
    # 转换为概率分布并采样
    probs = F.softmax(scaled_logits, dim=-1)
    sampled_token = torch.multinomial(probs, num_samples=1)
    return sampled_token.item()


def top_p_sampling(logits, p, temperature=1.0):
    """
    Top-P (Nucleus) 采样：保留累积概率刚好超过 p 的最小 token 集合
    
    Args:
        logits: 模型输出的原始 logits (vocab_size,)
        p: 累积概率阈值 (0 < p <= 1)
        temperature: 温度参数
    
    Returns:
        sampled_token: 采样得到的 token index
    """
    # 温度缩放
    scaled_logits = logits / temperature
    
    # 按概率从大到小排序
    sorted_logits, sorted_indices = torch.sort(scaled_logits, descending=True)
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    
    # 计算累积概率
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # 找到累积概率超过 p 的位置，将之后的 token 移除
    # 保留第一个超过 p 的位置（确保至少选 1 个 token）
    mask = cumulative_probs - sorted_probs > p
    sorted_logits[mask] = float('-inf')
    
    # 恢复原始顺序，转换为概率并采样
    # 创建与原始 logits 同形的张量
    filtered_logits = torch.full_like(logits, float('-inf'))
    filtered_logits.scatter_(0, sorted_indices, sorted_logits)
    
    probs = F.softmax(filtered_logits, dim=-1)
    sampled_token = torch.multinomial(probs, num_samples=1)
    return sampled_token.item()


# ============================================================
# 对比演示：Greedy vs Top-K vs Top-P
# ============================================================

print("=== 采样策略对比演示 ===\n")

# 使用之前定义的小型 Transformer 模型生成序列
# 复用 small_transformer（在 beam search 单元中创建的 128 维模型）

small_transformer.eval()
src_demo = torch.randint(3, 1000, (1, 8))  # 随机源序列

with torch.no_grad():
    encoder_out = small_transformer.encoder(src_demo)

def generate_with_strategy(decoder, encoder_output, strategy, max_len=12, **kwargs):
    """
    使用指定策略生成序列
    
    Args:
        decoder: 解码器
        encoder_output: 编码器输出
        strategy: 'greedy', 'top_k', 'top_p'
        max_len: 最大生成长度
        **kwargs: 策略参数 (k, p, temperature)
    """
    generated = [1]  # SOS token
    
    with torch.no_grad():
        for _ in range(max_len):
            tgt_tensor = torch.LongTensor([generated])
            tgt_mask = create_causal_mask(len(generated))
            output = decoder(tgt_tensor, encoder_output, tgt_mask=tgt_mask)
            logits = output[0, -1, :]  # 最后一个位置的 logits
            
            if strategy == 'greedy':
                next_token = logits.argmax().item()
            elif strategy == 'top_k':
                next_token = top_k_sampling(logits.clone(), k=kwargs.get('k', 10),
                                           temperature=kwargs.get('temperature', 1.0))
            elif strategy == 'top_p':
                next_token = top_p_sampling(logits.clone(), p=kwargs.get('p', 0.9),
                                           temperature=kwargs.get('temperature', 1.0))
            
            generated.append(next_token)
            if next_token == 2:  # EOS
                break
    
    return generated

# 用同一个输入，对比不同策略
print("源序列: (随机输入)\n")

# 贪婪解码（确定性，只运行一次）
greedy_seq = generate_with_strategy(small_transformer.decoder, encoder_out, 'greedy')
print(f"Greedy:       {greedy_seq}")

# Top-K 采样（随机，运行 3 次看多样性）
print(f"\nTop-K (k=10):")
for trial in range(3):
    seq = generate_with_strategy(small_transformer.decoder, encoder_out, 'top_k', k=10)
    print(f"  Trial {trial+1}: {seq}")

# Top-P 采样（随机，运行 3 次看多样性）
print(f"\nTop-P (p=0.9):")
for trial in range(3):
    seq = generate_with_strategy(small_transformer.decoder, encoder_out, 'top_p', p=0.9)
    print(f"  Trial {trial+1}: {seq}")

# 展示 Top-K 和 Top-P 对概率分布的影响
print("\n\n=== 概率分布裁剪效果 ===\n")

# 构造一个模拟的 logits 分布
torch.manual_seed(42)
demo_logits = torch.randn(50)  # 模拟 50 个 token 的 logits
demo_probs = F.softmax(demo_logits, dim=-1)
sorted_probs, sorted_idx = torch.sort(demo_probs, descending=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 原始分布
axes[0].bar(range(20), sorted_probs[:20].numpy(), color='steelblue', alpha=0.7)
axes[0].set_title('Original Distribution (Top-20)', fontweight='bold')
axes[0].set_xlabel('Token Rank')
axes[0].set_ylabel('Probability')

# Top-K (k=5) 裁剪
topk_probs = sorted_probs.clone()
topk_probs[5:] = 0
topk_probs = topk_probs / topk_probs.sum()
axes[1].bar(range(20), topk_probs[:20].numpy(), color='coral', alpha=0.7)
axes[1].set_title('After Top-K (k=5)', fontweight='bold')
axes[1].set_xlabel('Token Rank')

# Top-P (p=0.9) 裁剪
cumsum = torch.cumsum(sorted_probs, dim=0)
cutoff = (cumsum > 0.9).nonzero(as_tuple=True)[0][0].item() + 1
topp_probs = sorted_probs.clone()
topp_probs[cutoff:] = 0
topp_probs = topp_probs / topp_probs.sum()
axes[2].bar(range(20), topp_probs[:20].numpy(), color='seagreen', alpha=0.7)
axes[2].set_title(f'After Top-P (p=0.9, kept {cutoff} tokens)', fontweight='bold')
axes[2].set_xlabel('Token Rank')

plt.tight_layout()
plt.show()

print("观察：")
print("- Greedy 每次输出相同（确定性），可能重复单调")
print("- Top-K 固定保留 k 个候选词，多次采样结果不同（多样性）")
print("- Top-P 自适应保留候选词数量，分布尖锐时少选、平坦时多选")
print("- 实践中 Top-P=0.9 + Temperature=0.7~1.0 是常用组合")
print("\n✓ Top-K 与 Top-P 采样实现完成！")

## 7. 常见问题与调试 (FAQ & Debugging)

### Q1: Encoder-only vs Decoder-only vs Encoder-Decoder？

**A:** 三种架构适用于不同任务：

- **Encoder-only (BERT)**：理解任务（分类、NER）
- **Decoder-only (GPT)**：生成任务（文本生成、对话）
- **Encoder-Decoder (T5)**：转换任务（翻译、摘要）

### Q2: 如何处理不同长度的序列？

**A:** 
- **训练**：使用 padding 填充到相同长度，用 mask 忽略 padding
- **推理**：动态生成，直到 `<EOS>` 或达到最大长度

### Q3: Transformer 的内存复杂度是多少？

**A:** 
- 自注意力：$O(n^2 \cdot d)$，其中 $n$ 是序列长度
- 长序列会消耗大量内存
- 解决方案：稀疏注意力、局部注意力、Linformer 等

### Q4: 如何提高生成质量？

**A:**
- 使用束搜索而非贪婪解码
- 调整温度参数（temperature）
- 使用长度惩罚（length penalty）
- 增加模型容量和训练数据

### Q5: 为什么需要位置编码？

**A:** 
- Transformer 没有循环结构，无法感知位置
- 位置编码注入位置信息
- 使用正弦/余弦函数可以外推到更长序列

### 调试技巧

1. **检查掩码**：确保因果掩码正确应用
2. **监控注意力**：可视化注意力权重
3. **梯度检查**：使用梯度裁剪防止爆炸
4. **从小开始**：先用小模型和小数据验证

## 8. 总结与展望 (Summary)

### 核心要点

1. **掩码自注意力**：解码器使用因果掩码实现自回归生成
2. **交叉注意力**：解码器关注编码器输出，实现源-目标对齐
3. **解码器层**：三个子层（掩码自注意力、交叉注意力、FFN）
4. **完整 Transformer**：编码器-解码器架构用于序列到序列任务
5. **生成策略**：贪婪、束搜索、采样

### Transformer 家族

```
Transformer (2017)
    ↓
├─ Encoder-only: BERT (2018)
├─ Decoder-only: GPT (2018)
└─ Encoder-Decoder: T5 (2019)
    ↓
现代大语言模型
├─ GPT-3, GPT-4 (OpenAI)
├─ LLaMA (Meta)
├─ Claude (Anthropic)
└─ Gemini (Google)
```

### 与其他技术的联系

- **RNN/LSTM** → **Seq2Seq** → **Transformer**
- **注意力机制** → **自注意力** → **多头注意力**
- **Transformer** → **预训练模型** → **大语言模型**

### Module 3 回顾

本模块我们学习了：
1. ✅ **Self-Attention**：注意力机制的核心
2. ✅ **Transformer Encoder**：并行处理序列
3. ✅ **Transformer Decoder**：自回归生成

### 下一章预告

**Module 4: 预训练语言模型**
- 预训练任务（MLM, CLM）
- BERT 和 GPT 架构
- 微调（Fine-tuning）
- 迁移学习

### 💡 思考题

1. 为什么解码器需要掩码而编码器不需要？
2. 交叉注意力和自注意力的主要区别是什么？
3. 束搜索的 beam size 如何影响生成质量和速度？
4. Transformer 相比 RNN 的主要优势是什么？
5. 如何将 Transformer 应用到非序列数据（如图像）？

---

**恭喜！** 你已经掌握了完整的 Transformer 架构。

**下一步**：学习如何预训练和微调大规模语言模型。

## 思考题参考答案

### 问题 1：为什么解码器需要掩码而编码器不需要？

**答案：**

这一区别源于编码器和解码器所执行任务的根本不同。

**编码器的任务：理解（双向上下文）**

编码器负责"阅读理解"整个输入序列，其目标是为每个 token 生成包含全局上下文信息的表示。由于输入序列在推理时是完整可见的，每个 token 可以自由地关注序列中任意位置的 token（包括前向和后向），这样生成的表示更丰富、更准确。

例如，在理解句子 "The bank near the river" 中的 "bank" 时，模型需要同时看到后面的 "river" 才能判断这是"河岸"而非"银行"。双向注意力使这成为可能。

**解码器的任务：生成（自回归约束）**

解码器负责逐步生成目标序列，这是一个**自回归（Autoregressive）过程**：

$$P(y_1, y_2, ..., y_T | x) = \prod_{t=1}^{T} P(y_t | y_{<t}, x)$$

在生成第 $t$ 个 token 时，模型只知道前 $t-1$ 个已生成的 token，**不能提前看到**第 $t+1, t+2, ...$ 个 token（在推理时它们还不存在）。

**训练时的一致性问题：**

训练阶段使用 Teacher Forcing（用真实标签作为输入），效率上需要并行处理整个目标序列。若不加掩码，模型在训练时可以"偷看"未来的 token，从而"作弊"——直接复制未来信息而不是真正学习语言规律。因果掩码保证了训练和推理行为的一致性：

$$\text{mask}_{ij} = \begin{cases} 1 & \text{if } j \leq i \text{（允许关注过去和当前）} \\ 0 & \text{if } j > i \text{（禁止关注未来）} \end{cases}$$

**对比总结：**

| 特性 | 编码器 | 解码器 |
|------|--------|--------|
| 任务 | 理解输入序列 | 生成输出序列 |
| 注意力方向 | 双向（全部位置） | 单向（仅过去位置） |
| 是否需要掩码 | 不需要（可选 padding mask） | 需要因果掩码 |
| 典型代表 | BERT | GPT |
| 训练策略 | MLM（双向掩码语言模型） | CLM（因果语言模型） |

---

### 问题 2：交叉注意力和自注意力的主要区别是什么？

**答案：**

**自注意力（Self-Attention）：**

Query、Key、Value 全部来自**同一个序列**，用于捕获序列内部各 token 之间的关系：

$$Q = X W^Q, \quad K = X W^K, \quad V = X W^V$$

$$\text{Self-Attention}(X) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**交叉注意力（Cross-Attention）：**

Query 来自**解码器**（目标序列），Key 和 Value 来自**编码器输出**（源序列）：

$$Q = X_{dec} W^Q, \quad K = X_{enc} W^K, \quad V = X_{enc} W^V$$

$$\text{Cross-Attention}(X_{dec}, X_{enc}) = \text{softmax}\left(\frac{Q_{dec} K_{enc}^T}{\sqrt{d_k}}\right)V_{enc}$$

**核心区别对比：**

| 维度 | 自注意力 | 交叉注意力 |
|------|---------|----------|
| Q 来源 | 当前序列 | 解码器（目标序列） |
| K/V 来源 | 当前序列（与 Q 相同） | 编码器输出（源序列） |
| 注意力矩阵形状 | $(n \times n)$ 方阵 | $(m \times n)$ 矩形（$m$=目标长度，$n$=源长度） |
| 是否用因果掩码 | 解码器中需要 | 通常不需要 |
| 功能 | 序列内部关系建模 | 源-目标对齐建模 |
| 类比 | 阅读理解自己写的内容 | 参考外部资料写内容 |

**直觉理解：**

以机器翻译为例（英文 → 中文）：
- 解码器的**自注意力**：让当前已生成的中文词语之间相互关联（如"我"和"爱"的关系）
- **交叉注意力**：在生成每个中文词时，决定应该参考英文源句的哪些位置（类似于传统 Seq2Seq 中的对齐矩阵）

**可视化区别：**

```
自注意力矩阵（方阵，解码器内部）：       交叉注意力矩阵（矩形，跨序列）：
      T0  T1  T2                            S0  S1  S2  S3
T0  [ a   0   0  ]                  T0  [ a   b   c   d  ]
T1  [ e   f   0  ]                  T1  [ e   f   g   h  ]
T2  [ i   j   k  ]                  T2  [ i   j   k   l  ]
（下三角形，因果掩码）                （无掩码，可以看全部源序列）
```

---

### 问题 3：束搜索的 beam size 如何影响生成质量和速度？

**答案：**

**束搜索（Beam Search）工作原理：**

在每个时间步，不像贪婪解码只保留 1 个候选，束搜索保留 $k$（beam size）个最高分的候选序列，最终选择整体得分最高的序列。

**得分计算：**

$$\text{score}(y_{1:t}) = \sum_{i=1}^{t} \log P(y_i | y_{1:i-1}, x)$$

为防止偏好短序列，通常加入长度归一化：

$$\text{score\_norm}(y) = \frac{\text{score}(y)}{|y|^\alpha}, \quad \alpha \in [0, 1]$$

**beam size 的影响：**

**beam size = 1（退化为贪婪解码）：**
- 最快，每步只扩展 1 个候选
- 容易陷入局部最优，可能产生重复或语义不连贯的文本
- 适合：对速度要求极高、质量要求不高的场景

**beam size 增大（如 4, 8, 16）：**

| beam size | 生成质量 | 计算量 | 内存占用 | 适用场景 |
|-----------|---------|--------|---------|---------|
| 1 | 最低 | $O(n)$ | 最小 | 实时应用 |
| 4-5 | 较好 | $O(4n)$ | 中等 | 机器翻译（常见） |
| 10-20 | 更好 | $O(kn)$ | 较大 | 摘要生成 |
| >20 | 收益递减 | 很高 | 大 | 学术评测 |

**非单调关系（beam size 过大会变差）：**

在 NMT 任务中，Koehn & Knowles（2017）发现 beam size 超过某个阈值后，BLEU 分数反而下降，原因是：
- 束搜索偏好短句（即使有长度归一化）
- 大 beam size 容易找到"钻空子"的序列（语法正确但语义平凡）

**beam size 与任务类型：**

- **机器翻译**：beam size = 4-5，BLEU 提升显著
- **文本摘要**：beam size = 4-6，配合长度惩罚
- **对话生成**：不推荐束搜索，改用温度采样（diversity 更重要）
- **代码生成**：beam size = 1-10，取决于 pass@k 评测方式

**计算量分析：**

每步束搜索需要并行运行 $k$ 个前向传播，总计算量约为贪婪解码的 $k$ 倍，但由于现代 GPU 的批并行能力，实际速度差距通常小于 $k$ 倍（批大小为 $k$ 的推理效率高于逐个推理）。

---

### 问题 4：Transformer 相比 RNN 的主要优势是什么？

**答案：**

**全面对比：**

| 维度 | RNN/LSTM | Transformer |
|------|---------|-------------|
| **并行性** | 顺序依赖，无法并行 | 完全并行，训练快 10-100 倍 |
| **长程依赖** | 梯度消失，路径长度 $O(n)$ | 直接连接，路径长度 $O(1)$ |
| **训练时间** | $O(n)$（序列长度） | $O(1)$（理论上，受 GPU 限制） |
| **计算复杂度** | $O(n \cdot d^2)$ | $O(n^2 \cdot d + n \cdot d^2)$ |
| **内存复杂度** | $O(n \cdot d)$ | $O(n^2 + n \cdot d)$ |
| **参数数量** | 较少 | 较多 |
| **可解释性** | 较差 | 注意力可视化 |
| **扩展性** | 难以大规模扩展 | Scale 效果显著（Scaling Law）|

**核心优势详解：**

**1. 并行计算**

RNN 的隐藏状态依赖于前一时刻：$h_t = f(h_{t-1}, x_t)$，这使得整个序列只能串行处理。

Transformer 中，所有位置的注意力同时计算，充分利用 GPU 的并行能力：

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V \quad \text{（整体矩阵运算，完全并行）}$$

在 BERT 的训练中，Transformer 比同等参数量的 LSTM 快约 10 倍。

**2. 捕获长程依赖**

RNN 中，位置 $i$ 和位置 $j$（$j \gg i$）之间的信号需要经过 $j-i$ 步传递，梯度路径长，信息损耗大。

Transformer 任意两个位置之间的最大路径长度为 $O(1)$（通过注意力直接连接），无论序列多长，模型都能直接建立任意位置间的依赖关系。

**3. 可扩展性（Scaling Law）**

Kaplan et al.（2020）发现，Transformer 的性能遵循幂律关系随参数量、计算量、数据量的增加而持续提升，这一规律在 RNN 中不那么显著。这直接催生了 GPT-3、GPT-4、Claude 等大规模语言模型。

**RNN 的剩余优势（Transformer 的局限）：**

- **超长序列**：Transformer 的 $O(n^2)$ 内存使其难以处理极长序列（如百万 token），而 RNN 是 $O(n)$
- **流式推理**：RNN 天然支持增量处理（一次接收一个 token），Transformer 通常需要 KV Cache 优化
- **参数效率**：在小数据、低资源场景下，RNN 有时参数利用率更高

---

### 问题 5：如何将 Transformer 应用到非序列数据（如图像）？

**答案：**

将 Transformer 从序列数据扩展到图像等非序列数据的核心挑战是：如何将空间结构化数据转化为 Transformer 可以处理的"序列"，同时保留空间结构信息。

**方案一：ViT（Vision Transformer，最经典方案）**

Dosovitskiy et al.（2020）提出将图像分割成固定大小的 patch（如 $16 \times 16$ 像素），将每个 patch 展平后线性投影为 embedding，再拼接 `[CLS]` token：

```
图像 (224×224×3)
    ↓ 分成 14×14 = 196 个 patch（每个 16×16×3）
    ↓ 线性投影为 d_model 维向量
    ↓ 加上可学习的位置编码（2D 位置信息）
    ↓ 拼接 [CLS] token
    ↓ 输入标准 Transformer 编码器（197 × d_model）
    ↓ 取 [CLS] token 的输出做分类
```

**位置编码的修改：**

- 原始 ViT：可学习的 1D 位置嵌入（对 patch 序号编码）
- 改进方案：2D 正弦位置编码、相对位置编码（更好地捕获空间关系）

**方案二：Swin Transformer（层次化 + 局部注意力）**

针对 ViT 计算复杂度高的问题，Swin Transformer 引入：

1. **局部窗口注意力**：在 $7 \times 7$ 的局部窗口内计算注意力，复杂度从 $O(n^2)$ 降至 $O(n)$
2. **移位窗口（Shifted Window）**：交替移位窗口使不同窗口间的信息交流成为可能
3. **层次化特征图**：通过 patch merging 逐步下采样，类似 CNN 的多尺度特征

Swin Transformer 在多项计算机视觉任务（检测、分割）上超越了 CNN 方法。

**方案三：其他模态的迁移**

| 模态 | 处理方式 | 代表方法 |
|------|---------|---------|
| 图像 | Patch 分割 + 线性投影 | ViT, Swin |
| 视频 | 时空 patch（$t \times h \times w$） | VideoTransformer, TimeSformer |
| 点云 | 局部点集聚合为 token | Point Transformer |
| 音频 | 梅尔频谱图作为图像处理 | AST（Audio Spectrogram Transformer）|
| 图结构 | 节点嵌入 + 图位置编码 | Graphormer |
| 多模态 | 各模态 token 拼接后统一编码 | CLIP, DALL-E, Flamingo |

**关键设计要点：**

1. **如何分 token**：图像 patch、视频时空块、点云局部区域
2. **位置信息**：必须设计合适的位置编码来保留空间/时间结构
3. **计算效率**：高分辨率图像的 token 数量大，需要局部注意力或线性注意力
4. **预训练策略**：MAE（Masked Autoencoder）将 MLM 思想迁移到图像，遮蔽 75% 的 patch 进行自监督预训练

**多模态统一：**

近年来的研究表明，不同模态的数据都可以被视为"token 序列"，统一输入同一个 Transformer 架构进行处理（如 Gemini、GPT-4V），这正是 Transformer 强大通用性的体现。
